# Notebook 00 — YouTube Video & Transcript Extractor

**Purpose:** Download transcripts and metadata from any YouTube URL (video, Shorts, or playlist)  
**Output:** One `video_{id}.json` file per video saved to `../data/raw/`

## Pipeline position
```
00_video_extractor  ►  01_text_processing  ►  02_quality_analysis  ►  03_pinecone_upload  ►  04_agent_evaluation
```

> **Before running:** copy `.env.example` to `.env` and fill in your API keys.

## 01. Install dependencies

In [1]:
# Run this once if needed
#!pip install yt-dlp youtube-transcript-api

## 02.  Import libraries

In [2]:
import os
import json
import re
import time
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd

from youtube_transcript_api import YouTubeTranscriptApi
from yt_dlp import YoutubeDL
from tqdm.auto import tqdm

print('✅ Imports successful')

✅ Imports successful


In [3]:
# Configuration 
# Output directory for raw data
RAW_DATA_DIR = "../data/raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Transcript languages (in order of preference)
LANGUAGES = ("en",)

# Parallel processing settings
MAX_WORKERS = 4
SLEEP_BETWEEN_REQUESTS = 3  # seconds

print(f"📁 Raw data output directory: {RAW_DATA_DIR}")
print(f"🌍 Languages: {LANGUAGES}")
print(f"⚡ Max workers: {MAX_WORKERS}")

📁 Raw data output directory: ../data/raw
🌍 Languages: ('en',)
⚡ Max workers: 4


## 03. Create Helper functions

These helpers let you paste **one YouTube URL** and the notebook will detect whether it is a:
- normal YouTube video
- YouTube Shorts URL
- playlist URL

In [4]:
def detect_url_type(url: str) -> str:
    """
    Detect whether a YouTube URL is a video, short, or playlist.
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    query = parse_qs(parsed.query)
    path = parsed.path or ""

    if "list" in query:
        return "playlist"
    if "/shorts/" in path:
        return "short"
    if host == "youtu.be":
        return "video"
    if path == "/watch" and "v" in query:
        return "video"
    return "unknown"


def get_video_id(url: str) -> str:
    """
    Extract YouTube video ID from common URL formats:
    - https://www.youtube.com/watch?v=VIDEO_ID
    - https://youtu.be/VIDEO_ID
    - https://www.youtube.com/embed/VIDEO_ID
    - https://www.youtube.com/shorts/VIDEO_ID
    Also tolerates extra query params like ?si=...&list=...
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()

    if host == "youtu.be":
        vid = parsed.path.lstrip("/").split("/")[0]
        if vid and len(vid) == 11:
            return vid

    if host in {"www.youtube.com", "youtube.com", "m.youtube.com"}:
        if parsed.path == "/watch":
            q = parse_qs(parsed.query)
            vid = (q.get("v") or [None])[0]
            if vid and len(vid) == 11:
                return vid

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in {"embed", "shorts"}:
            vid = parts[1]
            if vid and len(vid) == 11:
                return vid
    # Fallback: regex scan
    match = re.search(r"([a-zA-Z0-9_-]{11})", url)
    if match:
        return match.group(1)

    raise ValueError(f"Could not extract a valid YouTube video ID from: {url}")

print('✅ Helper functions defined')

✅ Helper functions defined


## 04. Universal Youtube class collector

This version supports:
- one normal YouTube video URL
- one YouTube Shorts URL
- one playlist URL
- skipping videos that already exist on disk
- sequential mode
- parallel mode for larger playlists

In [5]:
class YouTubeCollector:
    """Collect YouTube video transcripts and metadata."""
    
    def __init__(self, output_dir=RAW_DATA_DIR, sleep_seconds=SLEEP_BETWEEN_REQUESTS):
        self.output_dir = output_dir
        self.sleep_seconds = sleep_seconds
        os.makedirs(self.output_dir, exist_ok=True)

# ── private helpers ──────────────────────────────────────────────────────

    def _build_ydl_options(self, extract_flat=True):
        return {
            "quiet": True,
            "no_warnings": True,
            "skip_download": True,
            "extract_flat": extract_flat,
            "ignoreerrors": True,
        }

    def _video_output_path(self, video_id: str) -> str:
        return os.path.join(self.output_dir, f"video_{video_id}.json")

    def _build_metadata(self, entry: dict) -> dict:
        """Extract metadata from yt-dlp entry."""
        video_id = entry.get("id") or entry.get("video_id")
        return {
            "video_id": video_id,
            "title": entry.get("title"),
            "url": f"https://www.youtube.com/watch?v={video_id}",
            "duration": entry.get("duration"),
            "channel": entry.get("uploader") or entry.get("channel"),
            "channel_id": entry.get("channel_id"),
            "upload_date": entry.get("upload_date"),
            "view_count": entry.get("view_count"),
            "like_count": entry.get("like_count"),
            "comment_count": entry.get("comment_count"),
            "description": entry.get("description"),
            "tags": entry.get("tags") or [],
            "categories": entry.get("categories") or [],
            "thumbnail": entry.get("thumbnail"),
            "playlist_index": entry.get("playlist_index"),
            "playlist_title": entry.get("playlist_title"),
        }

    def _fetch_transcript(self, video_id: str, languages=("en",)):
        """Fetch transcript for a video with diagnostics and fallbacks."""
        try:
            ytt_api = YouTubeTranscriptApi()
            transcript_list = ytt_api.list(video_id)

            available = []
            for t in transcript_list:
                available.append({
                    "language": t.language,
                    "language_code": t.language_code,
                    "is_generated": getattr(t, "is_generated", None),
                    "is_translatable": getattr(t, "is_translatable", None),
                })

            # 1) manual preferred
            try:
                transcript = transcript_list.find_manually_created_transcript(list(languages))
                fetched = transcript.fetch()
                return fetched.to_raw_data(), transcript.language_code, "manual", available
            except Exception as e_manual:
                manual_error = repr(e_manual)

            # 2) generated fallback
            try:
                transcript = transcript_list.find_generated_transcript(list(languages))
                fetched = transcript.fetch()
                return fetched.to_raw_data(), transcript.language_code, "auto-generated", available
            except Exception as e_generated:
                generated_error = repr(e_generated)

            # 3) any available transcript fallback
            try:
                any_transcript = next(iter(transcript_list))
                fetched = any_transcript.fetch()
                return fetched.to_raw_data(), any_transcript.language_code, "fallback_any_language", available
            except Exception as e_any:
                any_error = repr(e_any)

            return None, None, {
                "status": "not_available",
                "manual_error": manual_error,
                "generated_error": generated_error,
                "any_error": any_error,
                "available_transcripts": available,
            }, available

        except Exception as e:
            return None, None, {
                "status": "error",
                "error": repr(e),
            }, []

    def collect_single_video(self, url: str, languages=("en",), overwrite=False):
        """Collect data for a single video."""
        video_id = get_video_id(url)
        output_path = self._video_output_path(video_id)
        
        # Skip if already exists
        if not overwrite and os.path.exists(output_path):
            print(f"⏭️  Skipping {video_id} (already exists)")
            with open(output_path, "r", encoding="utf-8") as f:
                return json.load(f)
        
        print(f"📥 Collecting: {video_id}")
        
        # Get metadata
        with YoutubeDL(self._build_ydl_options(extract_flat=False)) as ydl:
            info = ydl.extract_info(url, download=False)
        
        metadata = self._build_metadata(info)
        
        # Get transcript
        transcript, lang, status, available = self._fetch_transcript(video_id, languages)
        
        # Build result
        result = {
            "video_id": video_id,
            "metadata": metadata,
            "transcript": transcript or [],
            "transcript_language": lang,
            "transcript_status": status,
            "available_transcripts": available,
            "collected_at": datetime.now().isoformat(),
        }
        
        # Save
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(f"   ✅ Title: {metadata.get('title')}")
        print(f"   ✅ Transcript: {len(transcript) if transcript else 0} segments ({status})")
        
        time.sleep(self.sleep_seconds)
        return result

    def get_playlist_entries(self, playlist_url: str, max_videos=None):
        """Get all video entries from a playlist."""
        with YoutubeDL(self._build_ydl_options(extract_flat=True)) as ydl:
            playlist_info = ydl.extract_info(playlist_url, download=False)
        
        entries = playlist_info.get("entries", [])
        if max_videos:
            entries = entries[:max_videos]
        
        # Add playlist context
        for idx, entry in enumerate(entries, 1):
            entry["playlist_index"] = idx
            entry["playlist_title"] = playlist_info.get("title")
            entry["url"] = f"https://www.youtube.com/watch?v={entry['id']}"
        
        return entries

    def collect_playlist(self, playlist_url: str, max_videos=None, 
                        languages=("en",), parallel=True, max_workers=4, overwrite=False):
        """Collect all videos from a playlist."""
        print(f"\n📋 Getting playlist entries...")
        entries = self.get_playlist_entries(playlist_url, max_videos)
        print(f"   Found {len(entries)} videos\n")
        
        if parallel and len(entries) > 1:
            return self._collect_parallel(entries, languages, max_workers, overwrite)
        else:
            return self._collect_sequential(entries, languages, overwrite)

    def _collect_sequential(self, entries, languages, overwrite):
        """Collect videos sequentially."""
        results = []
        for entry in tqdm(entries, desc="Collecting videos"):
            try:
                result = self.collect_single_video(entry["url"], languages, overwrite)
                results.append(result)
            except Exception as e:
                print(f"❌ Failed {entry['url']}: {e}")
        return results

    def _collect_parallel(self, entries, languages, max_workers, overwrite):
        """Collect videos in parallel."""
        results = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(self.collect_single_video, entry["url"], languages, overwrite): entry
                for entry in entries
            }
            
            with tqdm(total=len(entries), desc="Collecting videos") as pbar:
                for future in as_completed(futures):
                    entry = futures[future]
                    try:
                        result = future.result()
                        results.append(result)
                    except Exception as e:
                        print(f"\n❌ Failed {entry['url']}: {e}")
                    pbar.update(1)
        
        # Sort by playlist index
        results.sort(key=lambda x: x["metadata"].get("playlist_index") or 0)
        return results

    def collect_from_url(self, youtube_url: str, max_videos=None, 
                        languages=("en",), parallel=True, max_workers=4, overwrite=False):
        """Universal collector - handles video, short, or playlist URLs."""
        url_type = detect_url_type(youtube_url)
        print(f"🔍 Detected URL type: {url_type}")
        
        if url_type == "playlist":
            return self.collect_playlist(
                youtube_url, max_videos, languages, parallel, max_workers, overwrite
            )
        elif url_type in {"video", "short"}:
            return [self.collect_single_video(youtube_url, languages, overwrite)]
        else:
            raise ValueError(f"Unsupported URL type: {url_type}")

    def save_summary(self, results: list, filename="collection_summary.json"):
        """Save collection summary."""
        output_file = os.path.join(self.output_dir, filename)
        summary = {
            "total_videos": len(results),
            "generated_at": datetime.now().isoformat(),
            "videos": [
                {
                    "video_id": r["video_id"],
                    "title": r["metadata"].get("title"),
                    "url": r["metadata"].get("url"),
                    "transcript_status": r.get("transcript_status"),
                    "transcript_segments": len(r.get("transcript", [])),
                }
                for r in results
            ],
        }
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        print(f"\n📊 Summary saved to: {output_file}")
        return output_file


print("✅ YouTubeCollector class defined")

✅ YouTubeCollector class defined


## 05. Run Collection

### 📝 Instructions:
1. Paste your YouTube URL below (playlist, video, or short)
2. Adjust `max_videos` if needed (None = all videos)
3. Run the cell

### Examples:
```python
# Single video
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

# Playlist (recommended for this project)
youtube_url = "https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"

# YouTube Short
youtube_url = "https://www.youtube.com/shorts/VIDEO_ID"
```## Run the collector

In [18]:
# ⚠️ REPLACE WITH YOUR URL
youtube_url = "https://youtu.be/zSz0kV0BPDY?si=ny5hFlwoXawhU4mP"

# Configuration
max_videos = None  # Set to number like 25 to limit, or None for all
languages = ("en",)  # Preferred languages
parallel = True  # Use parallel processing for playlists
overwrite = False  # Set True to re-download existing videos

# Initialize collector
collector = YouTubeCollector(output_dir=RAW_DATA_DIR)

# Collect
results = collector.collect_from_url(
    youtube_url=youtube_url,
    max_videos=max_videos,
    languages=languages,
    parallel=parallel,
    max_workers=MAX_WORKERS,
    overwrite=overwrite,
)

print(f"\n✅ Collection complete! Processed {len(results)} video(s)")

🔍 Detected URL type: video
📥 Collecting: zSz0kV0BPDY
   ✅ Title: Why Are I-Beams Shaped Like An I?
   ✅ Transcript: 66 segments (manual)

✅ Collection complete! Processed 1 video(s)


## 06. Optional: save a summary file

Useful especially for playlists.

In [10]:
# summary_path = collector.save_summary(results, filename="collection_summary.json")
# print(f"Summary saved to: {summary_path}")

## 07. View Collection Statistics

In [11]:
import pandas as pd

# Build statistics
stats = []
for r in results:
    stats.append({
        "video_id": r["video_id"],
        "title": r["metadata"].get("title", "")[:50] + "...",
        "duration_sec": r["metadata"].get("duration"),
        "transcript_segments": len(r.get("transcript", [])),
        "transcript_status": r.get("transcript_status"),
        "channel": r["metadata"].get("channel"),
    })

df = pd.DataFrame(stats)
print(f"\n📊 Collection Statistics:")
print(f"   Total videos: {len(df)}")
print(f"   Total transcript segments: {df['transcript_segments'].sum()}")
print(f"   Videos with transcripts: {(df['transcript_segments'] > 0).sum()}")
print(f"\n")
display(df)


📊 Collection Statistics:
   Total videos: 1
   Total transcript segments: 191
   Videos with transcripts: 1




,video_id,title,duration_sec,transcript_segments,transcript_status,channel
0,6jQ4y0LK1kY,Heat Treatment -The Science of Forging (feat. ...,683,191,manual,Real Engineering


# 08. Verify output files

In [12]:
# List all collected video files
import glob

video_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "video_*.json")))
print(f"\n📁 Found {len(video_files)} video files in {RAW_DATA_DIR}")
print("\nFirst 5 files:")
for f in video_files[:5]:
    print(f"   {os.path.basename(f)}")


📁 Found 59 video files in ../data/raw

First 5 files:
   video_-CGxZ-qn4HA.json
   video_04K0bLwCDdM.json
   video_0VNIEfX0m4A.json
   video_1YTKedLQOa0.json
   video_21G7LA2DcGQ.json
